# Pan-ALDH × aldehyde docking pipeline

**Author:** Shen Yuchen (CMML3 ICA 2026)

This notebook documents the end-to-end pipeline used in the report.
Heavy steps (DiffDock inference, PyMOL combine, PRODIGY-LIG) run on the
GPU server (`/mnt/ShenYuchen/`); the resulting CSVs are read locally for
the analysis below.

## 1. Inputs
* `Data/proteinPDB/A1.pdb … A19.pdb, B1.pdb … B7.pdb`  — 26 ALDH structures.
* `Data/substrate/1_*.sdf … 60_*.sdf`                    — 60 aldehyde ligands.

## 2. Pipeline overview
1. `scripts/01_make_diffdock_csv.py` — build the 1560-row DiffDock CSV.
2. `python -m inference …`           — DiffDock-L on cuda.
3. `scripts/02_combine_pymol.py`    — chain A protein + chain L ligand combined PDB.
4. `scripts/03_run_prodigy.py`       — PRODIGY-LIG ΔG_noelec.
5. `scripts/04_distances.py`         — Cys-SG ↔ aldehyde-C distance.
6. `scripts/05_analysis.py`          — merge & plot.

Re-running the pipeline only requires `bash run_all.sh` on the GPU node.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DATA = Path('results')
df = pd.read_csv(DATA / 'master_results.csv')
df['ligand_num'] = df['ligand'].str.split('_', n=1).str[0].astype(int)
print(df.shape)
df.head()

In [ ]:
# Sanity: how many pairs reached productive geometry?
n_productive = (df['CYS_SG_to_aldC_distance_A'] <= 4).sum()
print(f'productive (≤4Å): {n_productive}/{len(df)}')
print(df['DG_prediction_kcalmol'].describe().round(2))

In [ ]:
# Top 15 strongest binders
top = df.sort_values('DG_prediction_kcalmol').head(15)
top[['protein','ligand','DG_prediction_kcalmol','CYS_SG_to_aldC_distance_A','rank1_confidence']]

In [ ]:
# Heat-map (Fig. 1)
from PIL import Image
Image.open(DATA / 'fig1_affinity_heatmap.png')

In [ ]:
# Distance vs ΔG (Fig. 2)
Image.open(DATA / 'fig2_dg_vs_distance.png')